<a href="https://colab.research.google.com/github/eyramg/ds2002-fa26/blob/main/2026_09_25_%E2%80%94_Cleaning_Gauntlet_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
dupes = df.duplicated().sum()

df = df.drop_duplicates().copy()

log('duplicates', 'dropped exact duplicate rows', dupes)

print('rows after dropping duplicates:', len(df))

[duplicates] dropped exact duplicate rows (15 row(s))
rows after dropping duplicates: 300


### TODO 2 — clean `price` -> float

In [4]:
df['price'] = (
    df['price']
    .str.replace('$', '', regex=False)
    .str.strip()
    .astype(float)
)

log('price', 'removed dollar signs and converted price to float', len(df))

print(df['price'].dtype)
print(df['price'].head())

[price] removed dollar signs and converted price to float (300 row(s))
float64
0    12.0
1     7.5
2     7.5
3     7.5
4     7.5
Name: price, dtype: float64


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')

missing_qty = df['qty'].isna().sum()
negative_qty = (df['qty'] < 0).sum()

df = df[df['qty'].notna() & (df['qty'] > 0)].copy()

log('qty', 'dropped rows with missing or negative quantity',
    missing_qty + negative_qty)

print('missing qty:', missing_qty)
print('negative qty:', negative_qty)
print('rows remaining:', len(df))

[qty] dropped rows with missing or negative quantity (25 row(s))
missing qty: 12
negative qty: 13
rows remaining: 275


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
print(df['item'].value_counts())

before = df['item'].nunique()

ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho'
}

df['item'] = df['item'].map(ITEM_MAP)

after = df['item'].nunique()

log('item',
    f'canonicalized item names from {before} variants to {after} products',
    len(df))

print(df['item'].value_counts())

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] canonicalized item names from 6 variants to 3 products (275 row(s))
item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
print('before:')
print(df['category'].value_counts())

CATEGORY_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'Merch': 'Merch',
    'Apparel': 'Apparel',
    'RainGear': 'Rain Gear',
    'rain-gear': 'Rain Gear'
}

before = df['category'].nunique()

df['category'] = df['category'].map(CATEGORY_MAP)

after = df['category'].nunique()

log('category',
    f'normalized {before} category labels to {after}; kept Apparel and Merch separate',
    len(df))

print('\nafter:')
print(df['category'].value_counts())

before:
category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category] normalized 6 category labels to 4; kept Apparel and Merch separate (275 row(s))

after:
category
Food         95
Rain Gear    86
Merch        51
Apparel      43
Name: count, dtype: int64


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float

assert df['item'].nunique() == 3
assert df['category'].nunique() == 4

print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
df['revenue'] = df['qty'] * df['price']

revenue_by_category = (
    df.groupby('category')['revenue']
    .sum()
    .sort_values(ascending=False)
    .round(2)
)

print(revenue_by_category)

overall_total = df['revenue'].sum()
print('Overall total:', round(overall_total, 2))

category
Food         1656.0
Rain Gear    1512.0
Merch         856.5
Apparel       715.5
Name: revenue, dtype: float64
Overall total: 4740.0


**What I would tell the vendor:** I would tell the vendor to stock more Food items because Food generated the highest revenue at $1,656.00.

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,dropped exact duplicate rows,15
1,price,removed dollar signs and converted price to float,300
2,qty,dropped rows with missing or negative quantity,25
3,item,canonicalized item names from 6 variants to 3 ...,275
4,category,normalized 6 category labels to 4; kept Appare...,275


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a) Dropping duplicate rows changed the revenue total the most. Revenue was $4,852.50 before removing the duplicates and $4,594.50 after removing them, a difference of $258.00.


b) One decision I made was keeping Apparel and Merch as separate categories. Another reasonable choice would be to combine them into one category. This would not change the overall revenue of $4,740.00, but their combined category revenue would be $1,572.00. I kept them separate because the original data treated Apparel and Merch as different categories.